# 03 — ML Baseline Models

**RAKSHAK-ICS** | Week 3 — Classical ML Baselines for ICS Anomaly Detection

This notebook runs all 7 ML baseline models on the preprocessed SWaT A9 dataset:

| # | Model | Type | AIML Unit |
|---|-------|------|-----------|
| 1 | Decision Tree | Supervised | Unit III |
| 2 | Random Forest | Supervised | Unit IV |
| 3 | KNN | Supervised | Unit IV |
| 4 | Naive Bayes | Supervised | Unit IV |
| 5 | Isolation Forest | Unsupervised | Unit V |
| 6 | K-Means | Unsupervised | Unit V |
| 7 | XGBoost | Supervised | Unit IV |

**Evaluation**: All results reported as **mean±std across 5 seeds** (42, 123, 456, 789, 1024).

> **Note**: SWaT A9 is a clean dataset (no attacks). We inject synthetic anomalies (5%) for evaluation.

In [ ]:
import os
os.chdir('..')

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
from pathlib import Path

plt.style.use('dark_background')
PALETTE = ['#00d4ff', '#ff6b6b', '#ffd93d', '#6bcb77', '#c084fc', '#ff922b', '#f472b6']
sns.set_palette(PALETTE)

print('Setup complete.')

## 1. Load Processed Data

In [ ]:
from src.preprocess import load_processed_data
from src.baselines import (
    MODEL_REGISTRY, get_model, flatten_windows,
    inject_synthetic_anomalies, compute_metrics,
    format_results_table, save_results
)
from src.stat_utils import run_with_seeds, format_result

data = load_processed_data('data/proof/')
X_train_w = data['X_train']
X_val_w = data['X_val']
X_test_w = data['X_test']

print(f'Train windows: {X_train_w.shape}')
print(f'Val windows:   {X_val_w.shape}')
print(f'Test windows:  {X_test_w.shape}')
print(f'Features:      {len(data["feature_names"])}')

## 2. Flatten Windows for Sklearn

In [ ]:
# Flatten 3D windows (N, 60, 65) -> 2D (N, 3900) for sklearn
X_train = flatten_windows(X_train_w)
X_val = flatten_windows(X_val_w)
X_test = flatten_windows(X_test_w)

print(f'Flattened train: {X_train.shape}')
print(f'Flattened val:   {X_val.shape}')
print(f'Flattened test:  {X_test.shape}')
print(f'Feature vector dimensionality: {X_train.shape[1]}')

## 3. Run All Baselines (5-Seed Evaluation)

For each model and each seed:
1. Inject synthetic anomalies (5%) into train/test
2. Train the model
3. Evaluate on test set
4. Collect F1, Precision, Recall, AUC-ROC

In [ ]:
SEEDS = [42, 123, 456, 789, 1024]
ANOMALY_FRACTION = 0.05
MODELS = list(MODEL_REGISTRY.keys())

all_results = {}

for model_name in MODELS:
    print(f'\n{"="*60}')
    print(f'Running {model_name}...')
    print(f'{"="*60}')
    
    seed_metrics = []
    
    for seed in SEEDS:
        # Inject anomalies with this seed
        X_train_aug, y_train = inject_synthetic_anomalies(X_train, ANOMALY_FRACTION, seed=seed)
        X_test_aug, y_test = inject_synthetic_anomalies(X_test, ANOMALY_FRACTION, seed=seed + 1000)
        
        # Train
        np.random.seed(seed)
        model = get_model(model_name)
        
        t0 = time.time()
        if model.model_type == 'supervised':
            model.fit(X_train_aug, y_train)
        else:
            model.fit(X_train_aug, y_train)  # unsupervised uses normal subset internally
        train_time = time.time() - t0
        
        # Evaluate
        y_pred = model.predict(X_test_aug)
        y_score = model.score(X_test_aug)
        metrics = compute_metrics(y_test, y_pred, y_score)
        metrics['train_time'] = train_time
        seed_metrics.append(metrics)
        
        print(f'  Seed {seed}: F1={metrics["f1"]:.4f}, AUC={metrics["auc_roc"]:.4f}, time={train_time:.1f}s')
    
    # Aggregate
    agg = {}
    for metric in ['f1', 'precision', 'recall', 'auc_roc', 'train_time']:
        values = [m[metric] for m in seed_metrics]
        agg[metric] = {
            'mean': float(np.mean(values)),
            'std': float(np.std(values)),
            'formatted': f'{np.mean(values):.4f}\u00b1{np.std(values):.4f}',
        }
    
    all_results[model_name] = {'aggregated': agg, 'per_seed': seed_metrics}
    print(f'  >> {model_name}: F1={agg["f1"]["formatted"]}, AUC={agg["auc_roc"]["formatted"]}')

print(f'\nAll {len(MODELS)} models complete!')

## 4. Results Comparison Table

In [ ]:
# Build comparison table
rows = []
for name, res in all_results.items():
    agg = res['aggregated']
    rows.append({
        'Model': name,
        'F1': agg['f1']['formatted'],
        'Precision': agg['precision']['formatted'],
        'Recall': agg['recall']['formatted'],
        'AUC-ROC': agg['auc_roc']['formatted'],
        'Train Time (s)': f"{agg['train_time']['mean']:.1f}",
    })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

## 5. F1 Score Comparison (Bar Chart)

In [ ]:
models = list(all_results.keys())
f1_means = [all_results[m]['aggregated']['f1']['mean'] for m in models]
f1_stds = [all_results[m]['aggregated']['f1']['std'] for m in models]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(models)), f1_means, yerr=f1_stds, capsize=5,
              color=PALETTE[:len(models)], edgecolor='white', linewidth=0.5, alpha=0.9)

ax.set_xticks(range(len(models)))
ax.set_xticklabels([m.replace('_', ' ').title() for m in models], fontsize=11, rotation=15)
ax.set_ylabel('F1 Score', fontsize=13)
ax.set_title('ML Baseline F1 Scores (mean \u00b1 std, 5 seeds)', fontsize=16, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

for i, (m, s) in enumerate(zip(f1_means, f1_stds)):
    ax.text(i, m + s + 0.02, f'{m:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/ml_baselines_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Multi-Metric Radar Chart

In [ ]:
metrics_list = ['f1', 'precision', 'recall', 'auc_roc']
n_metrics = len(metrics_list)
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for i, model_name in enumerate(models):
    values = [all_results[model_name]['aggregated'][m]['mean'] for m in metrics_list]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name.replace('_', ' ').title(),
            color=PALETTE[i % len(PALETTE)])
    ax.fill(angles, values, alpha=0.1, color=PALETTE[i % len(PALETTE)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([m.replace('_', ' ').upper() for m in metrics_list], fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('ML Baseline Multi-Metric Comparison', fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/figures/ml_baselines_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Training Time Comparison

In [ ]:
times = [all_results[m]['aggregated']['train_time']['mean'] for m in models]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(range(len(models)), times, color=PALETTE[:len(models)], edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(models)))
ax.set_yticklabels([m.replace('_', ' ').title() for m in models], fontsize=11)
ax.set_xlabel('Training Time (seconds)', fontsize=13)
ax.set_title('ML Baseline Training Times', fontsize=16, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for i, t in enumerate(times):
    ax.text(t + max(times)*0.01, i, f'{t:.1f}s', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('results/figures/ml_baselines_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Results

In [ ]:
# Save full results JSON
save_results(all_results, output_dir='results/tables/', filename='ml_baseline_results.json')

# Save CSV table
df_results.to_csv('results/tables/ml_baseline_results.csv', index=False)
print('Results saved to results/tables/')
print(f'  - ml_baseline_results.json')
print(f'  - ml_baseline_results.csv')

## 9. Key Findings

### Summary
- **All 7 ML baselines** evaluated with 5-seed reproducibility
- **Synthetic anomalies** (5%) injected since SWaT A9 is clean
- Results reported as **mean\u00b1std** with statistical significance
- These baselines serve as the **lower bound** for the LSTM-AE + GAT Blue Agent

### Next Steps
- `03b_ai_baselines.ipynb` — AI search attacker evaluation
- `03c_dl_baselines.ipynb` — Anomaly Transformer + USAD
- `04_lstm_autoencoder.ipynb` — LSTM-AE Blue Agent (Week 4)